## ad_performance_daily

In [ ]:
import json
import os
import time
import random
from datetime import datetime, timedelta, timezone
from urllib.parse import quote

import pg8000.native
import urllib3


# ============================================================
# CONFIG
# ============================================================
FB_API_VERSION = os.getenv("FB_API_VERSION", "v18.0")

INSIGHTS_BATCH_SIZE = int(os.getenv("INSIGHTS_BATCH_SIZE", "50"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
BASE_BACKOFF = float(os.getenv("BASE_BACKOFF", "1.0"))

# 전날(T-1)까지의 데이터를 수집하고 이후 값이 변경되면 UPSERT로 업데이트
CONFIRMED_LAG_DAYS = int(os.getenv("CONFIRMED_LAG_DAYS", "1"))  # T-1 기준 (어제까지)
BACKFILL_DAYS = int(os.getenv("BACKFILL_DAYS", "7"))

MIN_REMAINING_MS = int(os.getenv("MIN_REMAINING_MS", "120000"))
CLEANUP_ENABLED = os.getenv("CLEANUP_ENABLED", "false").lower() == "true"

# 0값도 DB에 저장하여 이후 업데이트 대상으로 유지
MIN_IMPRESSIONS = int(os.getenv("MIN_IMPRESSIONS", "0"))


# ============================================================
# MAIN
# ============================================================
def lambda_handler(event, context):
    try:
        access_token = os.environ["META_ACCESS_TOKEN"]
        db_host = os.environ["DB_HOST"]
        db_user = os.environ["DB_USER"]
        db_password = os.environ["DB_PASSWORD"]
        db_name = os.environ["DB_NAME"]

        # ============================================================
        # DATE RANGE: 전날(T-1)까지 수집, 이후 UPSERT로 갱신
        # ============================================================
        KST = timezone(timedelta(hours=9))
        now_kst = datetime.now(KST)

        end_date = (now_kst - timedelta(days=CONFIRMED_LAG_DAYS)).date()
        start_date = end_date - timedelta(days=BACKFILL_DAYS - 1)

        since = start_date.strftime("%Y-%m-%d")
        until = end_date.strftime("%Y-%m-%d")

        print(f"📅 Sync range (KST): {since} ~ {until}")
        print(f"⚙️ CONFIRMED_LAG_DAYS={CONFIRMED_LAG_DAYS} (end=yesterday), BACKFILL_DAYS={BACKFILL_DAYS}")

        ad_accounts = get_ad_accounts_from_db(db_host, db_user, db_password, db_name)
        print(f"✅ Found {len(ad_accounts)} ad accounts")

        db_conn = pg8000.native.Connection(
            user=db_user, password=db_password,
            host=db_host, database=db_name, port=5432
        )

        http = urllib3.PoolManager(
            timeout=urllib3.Timeout(connect=30.0, read=90.0),
            retries=False
        )

        total_saved = 0
        total_skipped_zero = 0
        total_api_errors = 0
        partial_run = False

        try:
            for fb_ad_account_id in ad_accounts:

                if context and context.get_remaining_time_in_millis() < MIN_REMAINING_MS:
                    print(f"⚠️ Lambda timeout approaching — partial run")
                    partial_run = True
                    break

                act_id = normalize_act_id(fb_ad_account_id)

                ads = get_ads_by_account_with_conn(db_conn, fb_ad_account_id)
                if not ads:
                    print(f"ℹ️ No ads found for account {fb_ad_account_id}, skipping")
                    continue

                ad_id_map = {row["fb_ad_id"]: row["ad_id"] for row in ads}
                fb_ad_ids = list(ad_id_map.keys())

                print(f"🔄 Account {act_id}: {len(fb_ad_ids)} ads")

                for batch in chunk_list(fb_ad_ids, INSIGHTS_BATCH_SIZE):

                    # ✅ [FIX 3] 페이지네이션 처리 추가 (이전엔 첫 페이지만 저장)
                    rows, api_error = fetch_account_insights_batch_all_pages(
                        http=http,
                        access_token=access_token,
                        act_id=act_id,
                        fb_ad_ids=batch,
                        since=since,
                        until=until
                    )

                    if api_error:
                        total_api_errors += 1

                    if rows:
                        saved, skipped = save_to_rds_daily_with_conn(
                            db_conn,
                            rows,
                            ad_id_map,
                            now_kst
                        )
                        total_saved += saved
                        total_skipped_zero += skipped

                    time.sleep(random.uniform(0.2, 0.5))

        finally:
            http.clear()
            db_conn.close()

        print(f"✅ Total saved: {total_saved} | Skipped (zero/invalid): {total_skipped_zero} | API errors: {total_api_errors}")

        return {
            "statusCode": 200 if not partial_run else 206,
            "body": json.dumps({
                "sync_range": {"since": since, "until": until},
                "confirmed_lag_days": CONFIRMED_LAG_DAYS,
                "backfill_days": BACKFILL_DAYS,
                "total_saved": total_saved,
                "total_skipped_zero": total_skipped_zero,
                "total_api_errors": total_api_errors,
                "partial_run": partial_run
            })
        }

    except Exception as e:
        import traceback
        traceback.print_exc()
        return {"statusCode": 500, "body": json.dumps({"error": str(e)})}


# ============================================================
# UTIL
# ============================================================
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def normalize_act_id(fb_ad_account_id):
    return fb_ad_account_id if fb_ad_account_id.startswith("act_") else f"act_{fb_ad_account_id}"


def exponential_backoff(attempt, base=BASE_BACKOFF, cap=60.0):
    """지수 백오프 + 지터"""
    delay = min(cap, base * (2 ** attempt))
    return delay + random.uniform(0, delay * 0.2)


# ============================================================
# DB
# ============================================================
def get_ad_accounts_from_db(host, user, password, database):
    conn = pg8000.native.Connection(user=user, password=password, host=host, database=database, port=5432)
    try:
        result = conn.run("SELECT fb_ad_account_id FROM ad_account WHERE fb_ad_account_id IS NOT NULL")
        return [str(r[0]).strip() for r in result]
    finally:
        conn.close()


def get_ads_by_account_with_conn(conn, fb_ad_account_id):
    result = conn.run("""
        SELECT ad_id, fb_ad_id
        FROM ad
        WHERE account_id = (
            SELECT account_id FROM ad_account WHERE fb_ad_account_id = :fb_ad_account_id
        )
        AND fb_ad_id IS NOT NULL
        AND status = 'ACTIVE'
    """, fb_ad_account_id=fb_ad_account_id)

    return [{"ad_id": r[0], "fb_ad_id": str(r[1]).strip()} for r in result]


# ============================================================
# META API
# ============================================================
def _build_insights_url(access_token, act_id, fb_ad_ids, since, until, after_cursor=None):
    """인사이트 API URL 생성 (페이지네이션 커서 포함)"""
    base_url = f"https://graph.facebook.com/{FB_API_VERSION}/{act_id}/insights"

    filtering = [{
        "field": "ad.id",
        "operator": "IN",
        "value": fb_ad_ids
    }]

    params = {
        "access_token": access_token,
        "level": "ad",
        "breakdowns": "age,gender",
        "time_range": json.dumps({"since": since, "until": until}),
        "time_increment": "1",
        "fields": "ad_id,ad_name,date_start,reach,impressions,clicks,ctr,inline_post_engagement",
        "filtering": json.dumps(filtering),
        "limit": "500",  # ✅ [FIX 3] 페이지 단위 축소 (5000→500, API 안정성 향상)
    }

    if after_cursor:
        params["after"] = after_cursor

    return base_url + "?" + "&".join([f"{k}={quote(str(v))}" for k, v in params.items()])


def fetch_account_insights_batch_all_pages(http, access_token, act_id, fb_ad_ids, since, until):
    """
    ✅ [FIX 3] 페이지네이션 완전 처리
    이전 코드는 첫 페이지(최대 5000건)만 가져왔음 → 데이터 유실 발생
    """
    all_rows = []
    after_cursor = None
    page_count = 0
    api_error = False
    max_pages = 50  # 무한루프 안전장치

    while page_count < max_pages:
        url = _build_insights_url(access_token, act_id, fb_ad_ids, since, until, after_cursor)

        data = None
        for attempt in range(MAX_RETRIES):
            try:
                resp = http.request("GET", url)
                data = json.loads(resp.data.decode("utf-8", errors="replace"))

                # ✅ [FIX 4] Rate limit 처리
                if resp.status == 429 or (
                    "error" in data and data["error"].get("code") in (4, 17, 32, 613)
                ):
                    wait = exponential_backoff(attempt)
                    print(f"⏳ Rate limited. Waiting {wait:.1f}s (attempt {attempt+1}/{MAX_RETRIES})")
                    time.sleep(wait)
                    continue

                if resp.status != 200 or "data" not in data:
                    print(f"⚠️ API error (status={resp.status}): {data.get('error', data)}")
                    api_error = True
                    data = None
                    break

                break  # 성공

            except Exception as e:
                wait = exponential_backoff(attempt)
                print(f"⚠️ Request exception: {e}. Retrying in {wait:.1f}s")
                time.sleep(wait)
                data = None

        if not data or "data" not in data:
            break

        rows = normalize_insights_rows(data["data"])
        all_rows.extend(rows)
        page_count += 1

        # ✅ 다음 페이지 커서 확인
        paging = data.get("paging", {})
        cursors = paging.get("cursors", {})
        next_url = paging.get("next")

        if next_url and cursors.get("after"):
            after_cursor = cursors["after"]
        else:
            break  # 마지막 페이지

    if page_count > 1:
        print(f"📄 Fetched {page_count} pages, {len(all_rows)} total rows")

    return all_rows, api_error


def normalize_insights_rows(data_rows):
    out = []
    for insight in data_rows or []:

        def to_int(v):
            try:
                return int(float(v))
            except Exception:
                return 0

        def to_float(v):
            try:
                return float(v)
            except Exception:
                return 0.0

        # ✅ [FIX 5] date_start 유효성 검사
        date_start = insight.get("date_start")
        if not date_start:
            continue

        out.append({
            "fb_ad_id": str(insight.get("ad_id", "")).strip(),
            "date_start": date_start,
            "age": insight.get("age") or "unknown",
            "gender": insight.get("gender") or "unknown",
            "reach": to_int(insight.get("reach")),
            "impressions": to_int(insight.get("impressions")),
            "clicks": to_int(insight.get("clicks")),
            "ctr": to_float(insight.get("ctr")),
            "inline_post_engagement": to_int(insight.get("inline_post_engagement")),
        })
    return out


# ============================================================
# SAVE
# ============================================================
def save_to_rds_daily_with_conn(db_conn, rows, ad_id_map, now_kst):
    """
    ✅ [FIX 2] 0값 필터 강화
    - 기존: 3개 필드 모두 0인 경우에만 skip → impressions=0 이어도 저장됨
    - 변경: impressions < MIN_IMPRESSIONS(1) 이면 skip (노출 없는 건 의미 없음)
    - fb_ad_id 없는 경우도 명시적으로 로깅
    """
    saved = 0
    skipped = 0

    for record in rows:

        # ✅ fb_ad_id 공백/빈값 방어
        fb_ad_id = record.get("fb_ad_id", "").strip()
        if not fb_ad_id:
            skipped += 1
            continue

        # ✅ ad_id_map에 없는 광고는 skip
        if fb_ad_id not in ad_id_map:
            skipped += 1
            continue

        # impressions가 0이어도 저장 (이후 API 업데이트 시 UPSERT로 갱신)
        if record["impressions"] < MIN_IMPRESSIONS:
            skipped += 1
            continue

        # ✅ [FIX 5] CTR 역산 검증: impressions > 0인데 clicks > impressions 이면 이상 데이터
        if record["clicks"] > record["impressions"]:
            print(f"⚠️ Suspicious row (clicks > impressions): ad={fb_ad_id}, date={record['date_start']}, "
                  f"clicks={record['clicks']}, impressions={record['impressions']} — skipping")
            skipped += 1
            continue

        try:
            db_conn.run("""
                INSERT INTO ad_performance_daily
                (ad_id, date, age, gender, reach, impressions, clicks, ctr,
                 inline_post_engagement, created_at, updated_at)
                VALUES
                (:ad_id, :date, :age, :gender, :reach, :impressions, :clicks, :ctr,
                 :inline_post_engagement, :created_at, :updated_at)
                ON CONFLICT (ad_id, date, age, gender)
                DO UPDATE SET
                    reach = EXCLUDED.reach,
                    impressions = EXCLUDED.impressions,
                    clicks = EXCLUDED.clicks,
                    ctr = EXCLUDED.ctr,
                    inline_post_engagement = EXCLUDED.inline_post_engagement,
                    updated_at = :updated_at
            """,
            ad_id=ad_id_map[fb_ad_id],
            date=record["date_start"],
            age=record["age"],
            gender=record["gender"],
            reach=record["reach"],
            impressions=record["impressions"],
            clicks=record["clicks"],
            ctr=record["ctr"],
            inline_post_engagement=record["inline_post_engagement"],
            created_at=now_kst,
            updated_at=now_kst)

            saved += 1

        except Exception as e:
            print(f"❌ DB insert error for ad={fb_ad_id}, date={record['date_start']}: {e}")
            skipped += 1

    return saved, skipped

## ad_performance_conversion_daily

In [ ]:
import json
import os
import random
import time
from datetime import datetime, timedelta, timezone
from typing import Dict, Iterable, List, Tuple, Optional

import pg8000.native
import requests

# =========================================================
# Config
# =========================================================
FB_API_VERSION = os.getenv("FB_API_VERSION", "v20.0")
BASE_URL = f"https://graph.facebook.com/{FB_API_VERSION}"
ACCESS_TOKEN = os.environ["META_ACCESS_TOKEN"]

DB_HOST = os.environ["DB_HOST"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_PORT = int(os.getenv("DB_PORT", "5432"))

# 최근 N일치 데이터를 수집하여 UPSERT (값 변경 시 자동 업데이트)
DAYS_BACK = int(os.getenv("DAYS_BACK", "3"))

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "50"))
API_SLEEP_BASE = float(os.getenv("API_SLEEP_BASE", "0.4"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "6"))
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "90"))

# If true, any per-account/day failure will fail the Lambda (recommended)
FAIL_ON_PARTIAL = os.getenv("FAIL_ON_PARTIAL", "true").strip().lower() in ("1", "true", "yes", "y")

KST = timezone(timedelta(hours=9))

ACTION_KEYS = {
    "link_clicks": ["link_click", "inline_link_click", "outbound_click", "outbound_clicks"],
    "add_to_cart": ["add_to_cart", "offsite_conversion.fb_pixel_add_to_cart", "omni_add_to_cart"],
    "website_landing_page_views": ["landing_page_view", "omni_landing_page_view"],
    "purchases": ["purchase", "offsite_conversion.fb_pixel_purchase", "omni_purchase"],
}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "ad-performance-conversion-daily/1.1"})


# =========================================================
# Helpers
# =========================================================
def chunked(items: List[str], size: int) -> Iterable[List[str]]:
    for i in range(0, len(items), size):
        yield items[i : i + size]


def normalize_act_id(value) -> str:
    if value is None:
        return ""
    s = str(value).strip()
    while s.startswith("act_"):
        s = s.replace("act_", "", 1)
    return s.strip()


def _sleep_backoff(attempt: int) -> None:
    delay = (API_SLEEP_BASE * (2**attempt)) + random.uniform(0, API_SLEEP_BASE)
    time.sleep(min(delay, 30.0))


def is_retryable_meta_error(code, msg: str) -> bool:
    m = (msg or "").lower()
    if code in (4, 17, 32, 613):
        return True
    if "rate" in m and "limit" in m:
        return True
    if "too many calls" in m or "user request limit" in m:
        return True
    if "temporarily unavailable" in m or "try again later" in m:
        return True
    return False


def graph_get(path: str, params: Dict) -> Dict:
    url = f"{BASE_URL}/{path.lstrip('/')}"
    request_params = dict(params or {})
    request_params["access_token"] = ACCESS_TOKEN

    last_error: Optional[Exception] = None

    for attempt in range(MAX_RETRIES):
        try:
            response = SESSION.get(url, params=request_params, timeout=REQUEST_TIMEOUT)
            payload_text = response.text

            if response.status_code >= 400:
                try:
                    payload = response.json()
                except Exception:
                    last_error = RuntimeError(f"HTTP {response.status_code}: {payload_text[:300]}")
                    _sleep_backoff(attempt)
                    continue

                err = payload.get("error") or {}
                code = err.get("code")
                subcode = err.get("error_subcode")
                msg = (err.get("message", "") or "")[:300]

                # non-retryable: invalid object
                if code == 100 and subcode == 33:
                    raise RuntimeError(
                        f"Meta API non-retryable error: code={code}, subcode={subcode}, msg={msg}"
                    )

                if is_retryable_meta_error(code, msg):
                    print(
                        f"Retryable Meta error code={code} subcode={subcode} "
                        f"attempt={attempt + 1}/{MAX_RETRIES}: {msg[:160]}"
                    )
                    _sleep_backoff(attempt)
                    continue

                raise RuntimeError(f"Meta API error: code={code}, subcode={subcode}, msg={msg}")

            return response.json()

        except RuntimeError as e:
            last_error = e
            if "non-retryable" in str(e).lower():
                raise
            if attempt == MAX_RETRIES - 1:
                raise
            _sleep_backoff(attempt)

        except Exception as e:
            last_error = e
            if attempt == MAX_RETRIES - 1:
                raise
            print(f"Network error {type(e).__name__}: {str(e)[:180]} | retry {attempt + 1}/{MAX_RETRIES}")
            _sleep_backoff(attempt)

    raise RuntimeError(f"graph_get failed: {last_error}")


def parse_actions_list(actions: List[Dict]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    for action in (actions or []):
        key = action.get("action_type")
        value = action.get("value")
        if key is None or value is None:
            continue
        try:
            out[key] = float(value)
        except Exception:
            continue
    return out


def parse_roas(roas_field):
    if roas_field is None:
        return None
    if isinstance(roas_field, (int, float)):
        return float(roas_field)
    if isinstance(roas_field, str):
        try:
            return float(roas_field)
        except Exception:
            return None
    if isinstance(roas_field, list):
        vals = []
        for x in roas_field:
            if not isinstance(x, dict):
                continue
            v = x.get("value")
            if v is None:
                continue
            try:
                vals.append(float(v))
            except Exception:
                continue
        return max(vals) if vals else None
    return None


def max_from_actions(actions_map: Dict[str, float], candidates: List[str]):
    vals = []
    for key in candidates:
        if key not in actions_map:
            continue
        try:
            vals.append(float(actions_map[key]))
        except Exception:
            continue
    return max(vals) if vals else None


def normalize_metrics(item: Dict) -> Dict:
    actions_map = parse_actions_list(item.get("actions"))
    return {
        "spend": float(item.get("spend") or 0.0),
        "purchase_roas": parse_roas(item.get("purchase_roas")),
        "link_clicks": int(max_from_actions(actions_map, ACTION_KEYS["link_clicks"]) or 0),
        "website_landing_page_views": int(
            max_from_actions(actions_map, ACTION_KEYS["website_landing_page_views"]) or 0
        ),
        "add_to_cart": int(max_from_actions(actions_map, ACTION_KEYS["add_to_cart"]) or 0),
        "purchases": int(max_from_actions(actions_map, ACTION_KEYS["purchases"]) or 0),
    }


def db_connect():
    return pg8000.native.Connection(
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        database=DB_NAME,
        port=DB_PORT,
        ssl_context=True,
    )


# =========================================================
# DB ops
# =========================================================
def fetch_accounts_and_ads(conn) -> Tuple[Dict[str, List[str]], Dict[str, int], List[int]]:
    sql = """
    SELECT aa.fb_ad_account_id, a.ad_id, a.fb_ad_id
    FROM ad a
    JOIN ad_account aa ON a.account_id = aa.account_id
    WHERE a.fb_ad_id IS NOT NULL
      AND aa.fb_ad_account_id IS NOT NULL
      AND a.status = 'ACTIVE'
    """
    rows = conn.run(sql)

    by_act: Dict[str, List[str]] = {}
    adid_map: Dict[str, int] = {}
    all_ad_ids: List[int] = []

    for fb_ad_account_id, ad_id, fb_ad_id in rows:
        act_num = normalize_act_id(fb_ad_account_id)
        if not act_num:
            continue
        fb_ad_id = str(fb_ad_id)
        by_act.setdefault(act_num, []).append(fb_ad_id)
        adid_map[fb_ad_id] = int(ad_id)
        all_ad_ids.append(int(ad_id))

    return by_act, adid_map, all_ad_ids


def delete_daily_window(conn, since: str, until_exclusive: str, ad_ids: List[int]) -> None:
    if not ad_ids:
        return
    sql = """
    DELETE FROM ad_performance_daily
    WHERE date >= :since
      AND date < :until_exclusive
      AND ad_id = ANY(CAST(:ad_ids AS BIGINT[]))
    """
    conn.run(sql, since=since, until_exclusive=until_exclusive, ad_ids=ad_ids)


def upsert_daily(conn, row: Dict) -> None:
    sql = """
    INSERT INTO ad_performance_daily (
      ad_id, date, age, gender,
      spend, purchase_roas, purchases,
      link_clicks, website_landing_page_views, add_to_cart,
      updated_at
    )
    VALUES (
      :ad_id, :date, :age, :gender,
      :spend, :purchase_roas, :purchases,
      :link_clicks, :website_landing_page_views, :add_to_cart,
      NOW()
    )
    ON CONFLICT (ad_id, date, age, gender)
    DO UPDATE SET
      spend = EXCLUDED.spend,
      purchase_roas = EXCLUDED.purchase_roas,
      purchases = EXCLUDED.purchases,
      link_clicks = EXCLUDED.link_clicks,
      website_landing_page_views = EXCLUDED.website_landing_page_views,
      add_to_cart = EXCLUDED.add_to_cart,
      updated_at = NOW()
    """
    conn.run(sql, **row)


# =========================================================
# Meta fetch
# =========================================================
def fetch_account_insights_daily(act_num: str, time_range: Dict, fb_ad_ids: List[str]) -> List[Dict]:
    path = f"act_{act_num}/insights"
    params = {
        "level": "ad",
        "time_range": json.dumps(time_range),
        "time_increment": "1",
        "breakdowns": "age,gender",
        "fields": "ad_id,date_start,spend,actions,purchase_roas",
        "filtering": json.dumps([
            {"field": "ad.id", "operator": "IN", "value": fb_ad_ids},
        ]),
        "limit": 5000,
    }

    data: List[Dict] = []
    after = None

    while True:
        if after:
            params["after"] = after
        payload = graph_get(path, params)
        data.extend(payload.get("data") or [])

        paging = payload.get("paging") or {}
        cursors = paging.get("cursors") or {}
        after = cursors.get("after")
        if not after:
            break

        time.sleep(API_SLEEP_BASE)

    return data


# =========================================================
# Loader
# =========================================================
def load_daily(
    conn,
    by_act: Dict[str, List[str]],
    adid_map: Dict[str, int],
    since: str,
    until_exclusive: str,
) -> Tuple[int, List[str]]:
    """
    Loads daily insights for each day in [since, until_exclusive).
    IMPORTANT:
      - We DO NOT trust item['date_start'] for DB date.
      - We force the DB date to the requested day (day_since) to prevent timezone drift.
    Returns: (upserted_rows, failures)
    """
    start_date = datetime.fromisoformat(since).date()
    end_date = datetime.fromisoformat(until_exclusive).date() - timedelta(days=1)

    total_rows = 0
    failures: List[str] = []

    day = start_date
    while day <= end_date:
        day_since = day.strftime("%Y-%m-%d")
        day_until = (day + timedelta(days=1)).strftime("%Y-%m-%d")
        print(f"📅 Fetching daily insights date={day_since} (until={day_until}) | acts={len(by_act)}")

        for act_num, fb_ads in by_act.items():
            for fb_chunk in chunked(fb_ads, BATCH_SIZE):
                try:
                    insights = fetch_account_insights_daily(
                        act_num,
                        {"since": day_since, "until": day_until},
                        fb_chunk,
                    )
                except RuntimeError as e:
                    msg = str(e)
                    key = f"act={act_num} date={day_since} err={msg[:160]}"
                    print(f"❌ Skip {key}")
                    failures.append(key)
                    continue

                for item in insights:
                    fb_ad_id = str(item.get("ad_id"))
                    ad_id = adid_map.get(fb_ad_id)
                    if not ad_id:
                        continue

                    row = {
                        "ad_id": ad_id,
                        "date": day_since,  # ✅ 강제 (date_start 신뢰하지 않음)
                        "age": item.get("age") or "unknown",
                        "gender": item.get("gender") or "unknown",
                        **normalize_metrics(item),
                    }
                    upsert_daily(conn, row)
                    total_rows += 1

            time.sleep(API_SLEEP_BASE)

        day += timedelta(days=1)

    return total_rows, failures


# =========================================================
# Main
# =========================================================
def run() -> Dict:
    """
    Default behavior:
      - Process "yesterday" only (KST), unless DAYS_BACK > 1
      - Delete only the processing window [since, until_exclusive)
      - Fail the Lambda if partial failures happened (FAIL_ON_PARTIAL=true)
    """
    now_kst = datetime.now(KST)

    # End at yesterday (inclusive) => until_exclusive is yesterday+1
    end_day = (now_kst - timedelta(days=1)).date()
    start_day = end_day - timedelta(days=max(DAYS_BACK, 1) - 1)

    since = start_day.strftime("%Y-%m-%d")
    until_exclusive = (end_day + timedelta(days=1)).strftime("%Y-%m-%d")

    print(
        f"🚀 job=ad_performance_conversion_daily "
        f"now_kst={now_kst.isoformat()} "
        f"window=[{since}, {until_exclusive}) "
        f"DAYS_BACK={DAYS_BACK} "
        f"BATCH_SIZE={BATCH_SIZE} "
        f"FAIL_ON_PARTIAL={FAIL_ON_PARTIAL}"
    )

    conn = db_connect()
    try:
        by_act, adid_map, all_ad_ids = fetch_accounts_and_ads(conn)
        print(f"✅ Loaded ads: acts={len(by_act)} ads={len(adid_map)}")

        if not adid_map:
            result = {
                "job": "ad_performance_conversion_daily",
                "since": since,
                "until_exclusive": until_exclusive,
                "acts": len(by_act),
                "ads": 0,
                "upserted_rows": 0,
                "failures": 0,
                "note": "no ads found",
            }
            print(json.dumps(result, ensure_ascii=False))
            return result

        # Delete only the window we will rebuild (safe + minimal)
        delete_daily_window(conn, since=since, until_exclusive=until_exclusive, ad_ids=all_ad_ids)
        print(f"🧹 Deleted existing rows in window=[{since},{until_exclusive}) for ads={len(all_ad_ids)}")

        upserted_rows, failures = load_daily(
            conn,
            by_act=by_act,
            adid_map=adid_map,
            since=since,
            until_exclusive=until_exclusive,
        )

        result = {
            "job": "ad_performance_conversion_daily",
            "since": since,
            "until_exclusive": until_exclusive,
            "acts": len(by_act),
            "ads": len(adid_map),
            "upserted_rows": upserted_rows,
            "failures": len(failures),
            "sample_failure": failures[0] if failures else None,
        }
        print(json.dumps(result, ensure_ascii=False))

        # If any partial failure occurred, fail the Lambda (so EventBridge / Alarm can catch)
        if FAIL_ON_PARTIAL and failures:
            raise RuntimeError(
                f"Partial failures occurred ({len(failures)}). "
                f"Example: {failures[0]}. "
                f"Set FAIL_ON_PARTIAL=false to ignore."
            )

        return result

    finally:
        conn.close()


def lambda_handler(event, context):
    _ = event, context
    return run()


if __name__ == "__main__":
    run()

## Full Scan (전체 기간 DB 점검)

In [2]:
import json
import os
import random
import time
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Set, Tuple

import pg8000.native
import requests
from dotenv import load_dotenv

load_dotenv()

# =========================================================
# Config
# =========================================================
FB_API_VERSION = os.getenv("FB_API_VERSION", "v20.0")
BASE_URL = f"https://graph.facebook.com/{FB_API_VERSION}"
ACCESS_TOKEN = os.environ["META_ACCESS_TOKEN"]

DB_HOST = os.environ["DB_HOST"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_PORT = int(os.getenv("DB_PORT", "5432"))

KST = timezone(timedelta(hours=9))

# ▼ 점검할 날짜 범위 직접 지정 (환경변수 또는 아래 직접 수정)
START_DATE = os.getenv("START_DATE", "2024-01-01")
END_DATE = os.getenv("END_DATE", (datetime.now(KST) - timedelta(days=1)).strftime("%Y-%m-%d"))

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "50"))
API_SLEEP_BASE = float(os.getenv("API_SLEEP_BASE", "0.5"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "90"))

FLOAT_TOLERANCE = 1e-6  # float 비교 허용 오차

# 전환 지표 action type 매핑
ACTION_KEYS = {
    "link_clicks": ["link_click", "inline_link_click", "outbound_click", "outbound_clicks"],
    "add_to_cart": ["add_to_cart", "offsite_conversion.fb_pixel_add_to_cart", "omni_add_to_cart"],
    "website_landing_page_views": ["landing_page_view", "omni_landing_page_view"],
    "purchases": ["purchase", "offsite_conversion.fb_pixel_purchase", "omni_purchase"],
}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "ad-performance-full-scan/1.0"})


# =========================================================
# Helpers
# =========================================================
def chunked(items: List, size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def normalize_act_id(value) -> str:
    s = str(value or "").strip()
    while s.startswith("act_"):
        s = s[4:]
    return s.strip()


def _sleep_backoff(attempt: int, base: float = API_SLEEP_BASE) -> None:
    delay = min(base * (2 ** attempt) + random.uniform(0, base), 60.0)
    time.sleep(delay)


def is_permission_error(code, msg: str) -> bool:
    """계정 접근 권한 오류 여부 판별"""
    if code in (10, 200, 273, 294):
        return True
    m = (msg or "").lower()
    return any(k in m for k in ("permission", "authorized", "authorization", "does not have"))


def is_retryable_error(code, msg: str) -> bool:
    """재시도 가능한 일시적 오류 여부 판별 (rate limit 등)"""
    if code in (4, 17, 32, 613):
        return True
    m = (msg or "").lower()
    return any(k in m for k in ("rate", "too many", "temporarily", "try again"))


# =========================================================
# Meta API
# =========================================================
def graph_get(path: str, params: Dict) -> Tuple[Optional[Dict], Optional[int], Optional[str]]:
    """
    Returns (payload, err_code, err_msg).
    권한 오류는 즉시 반환 (caller가 재시도 여부 결정).
    Rate limit은 내부에서 재시도.
    """
    url = f"{BASE_URL}/{path.lstrip('/')}"
    req_params = {**params, "access_token": ACCESS_TOKEN}
    last_exc = None

    for attempt in range(MAX_RETRIES):
        try:
            resp = SESSION.get(url, params=req_params, timeout=REQUEST_TIMEOUT)
            payload = resp.json()

            if resp.status_code >= 400 or "error" in payload:
                err = payload.get("error") or {}
                code = err.get("code")
                msg = err.get("message", "")

                if is_permission_error(code, msg):
                    return None, code, msg  # 권한 오류: caller에게 위임

                if is_retryable_error(code, msg):
                    print(f"    ⏳ Rate limit (code={code}), retry {attempt + 1}/{MAX_RETRIES}")
                    _sleep_backoff(attempt)
                    continue

                return None, code, msg  # 기타 API 오류

            return payload, None, None  # 성공

        except Exception as e:
            last_exc = e
            print(f"    ⚠️ Network error: {e}, retry {attempt + 1}/{MAX_RETRIES}")
            _sleep_backoff(attempt)

    raise RuntimeError(f"graph_get failed after {MAX_RETRIES} retries: {last_exc}")


def fetch_insights_for_account(
    act_num: str, fb_ad_ids: List[str], since: str, until: str
) -> Tuple[List[Dict], Optional[int], Optional[str]]:
    """
    한 계정의 인사이트를 페이지네이션으로 전량 수집.
    모든 성과 지표(노출·클릭·전환·지출 등)를 단일 API 호출로 수집.
    Returns: (rows, perm_err_code, perm_err_msg)
    """
    path = f"act_{act_num}/insights"
    params = {
        "level": "ad",
        "time_range": json.dumps({"since": since, "until": until}),
        "time_increment": "1",
        "breakdowns": "age,gender",
        "fields": (
            "ad_id,date_start,"
            "reach,impressions,clicks,ctr,inline_post_engagement,"  # ad_performance_daily
            "spend,actions,purchase_roas"                            # ad_performance_conversion_daily
        ),
        "filtering": json.dumps([{"field": "ad.id", "operator": "IN", "value": fb_ad_ids}]),
        "limit": 500,
    }

    all_data: List[Dict] = []
    after = None

    while True:
        if after:
            params["after"] = after

        payload, err_code, err_msg = graph_get(path, params)
        if payload is None:
            return [], err_code, err_msg  # 오류 (권한 포함)

        all_data.extend(payload.get("data") or [])

        cursors = (payload.get("paging") or {}).get("cursors") or {}
        after = cursors.get("after")
        if not after:
            break

        time.sleep(API_SLEEP_BASE)

    return all_data, None, None


# =========================================================
# Normalize API row
# =========================================================
def parse_actions(actions: List[Dict]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    for a in (actions or []):
        k, v = a.get("action_type"), a.get("value")
        if k and v is not None:
            try:
                out[k] = float(v)
            except Exception:
                pass
    return out


def parse_roas(roas_field) -> Optional[float]:
    if roas_field is None:
        return None
    if isinstance(roas_field, (int, float)):
        return float(roas_field)
    if isinstance(roas_field, str):
        try:
            return float(roas_field)
        except Exception:
            return None
    if isinstance(roas_field, list):
        vals = [float(x["value"]) for x in roas_field if isinstance(x, dict) and x.get("value") is not None]
        return max(vals) if vals else None
    return None


def max_from_actions(actions_map: Dict[str, float], candidates: List[str]) -> int:
    vals = [actions_map[k] for k in candidates if k in actions_map]
    return int(max(vals)) if vals else 0


def normalize_item(item: Dict) -> Optional[Dict]:
    """API 응답 row를 DB 저장 형태로 정규화"""
    def to_int(v):
        try:
            return int(float(v))
        except Exception:
            return 0

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return 0.0

    fb_ad_id = str(item.get("ad_id", "")).strip()
    date_str = item.get("date_start")
    if not fb_ad_id or not date_str:
        return None

    actions_map = parse_actions(item.get("actions"))

    return {
        "fb_ad_id": fb_ad_id,
        "date": date_str,
        "age": item.get("age") or "unknown",
        "gender": item.get("gender") or "unknown",
        # ad_performance_daily 지표
        "reach": to_int(item.get("reach")),
        "impressions": to_int(item.get("impressions")),
        "clicks": to_int(item.get("clicks")),
        "ctr": to_float(item.get("ctr")),
        "inline_post_engagement": to_int(item.get("inline_post_engagement")),
        # ad_performance_conversion_daily 지표
        "spend": to_float(item.get("spend")),
        "purchase_roas": parse_roas(item.get("purchase_roas")),
        "link_clicks": max_from_actions(actions_map, ACTION_KEYS["link_clicks"]),
        "website_landing_page_views": max_from_actions(actions_map, ACTION_KEYS["website_landing_page_views"]),
        "add_to_cart": max_from_actions(actions_map, ACTION_KEYS["add_to_cart"]),
        "purchases": max_from_actions(actions_map, ACTION_KEYS["purchases"]),
    }


# =========================================================
# DB
# =========================================================
def db_connect():
    return pg8000.native.Connection(
        user=DB_USER, password=DB_PASSWORD,
        host=DB_HOST, database=DB_NAME,
        port=DB_PORT, ssl_context=True,
    )


def fetch_accounts_and_ads(conn) -> Tuple[Dict[str, List[str]], Dict[str, int]]:
    rows = conn.run("""
        SELECT aa.fb_ad_account_id, a.ad_id, a.fb_ad_id
        FROM ad a
        JOIN ad_account aa ON a.account_id = aa.account_id
        WHERE a.fb_ad_id IS NOT NULL
          AND aa.fb_ad_account_id IS NOT NULL
          AND a.status = 'ACTIVE'
    """)
    by_act: Dict[str, List[str]] = {}
    adid_map: Dict[str, int] = {}
    for fb_act_id, ad_id, fb_ad_id in rows:
        act_num = normalize_act_id(fb_act_id)
        if not act_num:
            continue
        fb_ad_id_str = str(fb_ad_id).strip()
        by_act.setdefault(act_num, []).append(fb_ad_id_str)
        adid_map[fb_ad_id_str] = int(ad_id)
    return by_act, adid_map


def load_db_snapshot(conn, ad_ids: List[int], since: str, until_inclusive: str) -> Dict[Tuple, Dict]:
    """
    DB에서 해당 기간 데이터를 메모리에 로드.
    Key: (ad_id, date_str, age, gender)
    """
    if not ad_ids:
        return {}
    rows = conn.run("""
        SELECT ad_id, date::text, age, gender,
               reach, impressions, clicks, ctr, inline_post_engagement,
               spend, purchase_roas, link_clicks, website_landing_page_views,
               add_to_cart, purchases
        FROM ad_performance_daily
        WHERE date >= :since
          AND date <= :until
          AND ad_id = ANY(CAST(:ad_ids AS BIGINT[]))
    """, since=since, until=until_inclusive, ad_ids=ad_ids)

    snapshot: Dict[Tuple, Dict] = {}
    for row in rows:
        (ad_id, date_str, age, gender,
         reach, impressions, clicks, ctr, inline_post_engagement,
         spend, purchase_roas, link_clicks, website_landing_page_views,
         add_to_cart, purchases) = row
        key = (int(ad_id), str(date_str), str(age), str(gender))
        snapshot[key] = {
            "reach": reach or 0,
            "impressions": impressions or 0,
            "clicks": clicks or 0,
            "ctr": float(ctr or 0),
            "inline_post_engagement": inline_post_engagement or 0,
            "spend": float(spend or 0),
            "purchase_roas": float(purchase_roas) if purchase_roas is not None else None,
            "link_clicks": link_clicks or 0,
            "website_landing_page_views": website_landing_page_views or 0,
            "add_to_cart": add_to_cart or 0,
            "purchases": purchases or 0,
        }
    return snapshot


def float_eq(a, b) -> bool:
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return abs(float(a) - float(b)) <= FLOAT_TOLERANCE


def rows_differ(db_row: Dict, api_row: Dict) -> bool:
    """DB 값과 API 값이 하나라도 다르면 True"""
    int_fields = [
        "reach", "impressions", "clicks", "inline_post_engagement",
        "link_clicks", "website_landing_page_views", "add_to_cart", "purchases",
    ]
    for f in int_fields:
        if (db_row.get(f) or 0) != (api_row.get(f) or 0):
            return True
    return (
        not float_eq(db_row.get("ctr"), api_row.get("ctr"))
        or not float_eq(db_row.get("spend"), api_row.get("spend"))
        or not float_eq(db_row.get("purchase_roas"), api_row.get("purchase_roas"))
    )


def upsert_row(conn, ad_id: int, row: Dict) -> None:
    conn.run("""
        INSERT INTO ad_performance_daily (
            ad_id, date, age, gender,
            reach, impressions, clicks, ctr, inline_post_engagement,
            spend, purchase_roas, link_clicks, website_landing_page_views,
            add_to_cart, purchases,
            created_at, updated_at
        ) VALUES (
            :ad_id, :date, :age, :gender,
            :reach, :impressions, :clicks, :ctr, :inline_post_engagement,
            :spend, :purchase_roas, :link_clicks, :website_landing_page_views,
            :add_to_cart, :purchases,
            NOW(), NOW()
        )
        ON CONFLICT (ad_id, date, age, gender) DO UPDATE SET
            reach                    = EXCLUDED.reach,
            impressions              = EXCLUDED.impressions,
            clicks                   = EXCLUDED.clicks,
            ctr                      = EXCLUDED.ctr,
            inline_post_engagement   = EXCLUDED.inline_post_engagement,
            spend                    = EXCLUDED.spend,
            purchase_roas            = EXCLUDED.purchase_roas,
            link_clicks              = EXCLUDED.link_clicks,
            website_landing_page_views = EXCLUDED.website_landing_page_views,
            add_to_cart              = EXCLUDED.add_to_cart,
            purchases                = EXCLUDED.purchases,
            updated_at               = NOW()
    """,
        ad_id=ad_id, date=row["date"], age=row["age"], gender=row["gender"],
        reach=row["reach"], impressions=row["impressions"],
        clicks=row["clicks"], ctr=row["ctr"],
        inline_post_engagement=row["inline_post_engagement"],
        spend=row["spend"], purchase_roas=row["purchase_roas"],
        link_clicks=row["link_clicks"],
        website_landing_page_views=row["website_landing_page_views"],
        add_to_cart=row["add_to_cart"], purchases=row["purchases"],
    )


# =========================================================
# Full scan
# =========================================================
def run_full_scan(start_date: str = START_DATE, end_date: str = END_DATE) -> Dict:
    """
    start_date ~ end_date 전체 기간에 대해 Meta API 데이터와 DB를 비교하여 점검.
    - DB에 없는 row → INSERT
    - DB 값과 다른 row → UPDATE
    - 동일한 row → SKIP
    - 계정 권한 오류 → 1회 재시도 후 계정 전체 스킵
    """
    print(f"🔍 Full scan: {start_date} ~ {end_date}")

    conn = db_connect()
    try:
        by_act, adid_map = fetch_accounts_and_ads(conn)
        all_ad_ids = list(set(adid_map.values()))
        print(f"✅ Acts: {len(by_act)} | Ads: {len(adid_map)}")

        # DB 스냅샷을 메모리에 미리 로드 (비교 기준)
        print(f"📂 Loading DB snapshot ({start_date} ~ {end_date})...")
        db_snapshot = load_db_snapshot(conn, all_ad_ids, start_date, end_date)
        print(f"   → {len(db_snapshot):,} existing rows loaded")

        skipped_accounts: Set[str] = set()
        total_checked = 0
        total_inserted = 0
        total_updated = 0
        total_skipped_same = 0
        total_skipped_perm_accts = 0

        for act_num, fb_ad_ids in by_act.items():
            if act_num in skipped_accounts:
                continue

            print(f"\n🔄 act_{act_num}: {len(fb_ad_ids)} ads")
            account_api_rows: List[Dict] = []
            account_had_perm_error = False
            perm_retry_done = False

            for fb_chunk in chunked(fb_ad_ids, BATCH_SIZE):
                rows, err_code, err_msg = fetch_insights_for_account(
                    act_num, fb_chunk, start_date, end_date
                )

                if err_code is not None and is_permission_error(err_code, err_msg):
                    if not perm_retry_done:
                        print(f"  ⚠️ Permission error (code={err_code}). 5초 후 재시도...")
                        time.sleep(5)
                        rows, err_code, err_msg = fetch_insights_for_account(
                            act_num, fb_chunk, start_date, end_date
                        )
                        perm_retry_done = True

                    if err_code is not None and is_permission_error(err_code, err_msg):
                        print(f"  ❌ 권한 오류 지속 (act_{act_num}). 계정 전체 스킵.")
                        skipped_accounts.add(act_num)
                        total_skipped_perm_accts += 1
                        account_had_perm_error = True
                        break

                account_api_rows.extend(rows)
                time.sleep(API_SLEEP_BASE)

            if account_had_perm_error:
                continue

            # API 결과 → DB 비교 및 업데이트
            for item in account_api_rows:
                row = normalize_item(item)
                if row is None:
                    continue

                fb_ad_id = row["fb_ad_id"]
                if fb_ad_id not in adid_map:
                    continue

                ad_id = adid_map[fb_ad_id]
                key = (ad_id, row["date"], row["age"], row["gender"])
                total_checked += 1

                db_row = db_snapshot.get(key)

                if db_row is None:
                    upsert_row(conn, ad_id, row)
                    db_snapshot[key] = row
                    total_inserted += 1
                elif rows_differ(db_row, row):
                    upsert_row(conn, ad_id, row)
                    db_snapshot[key] = row
                    total_updated += 1
                else:
                    total_skipped_same += 1

            print(f"  → inserted={total_inserted} updated={total_updated} "
                  f"same={total_skipped_same} checked={total_checked}")

        print(f"\n{'=' * 60}")
        print(f"✅ Full scan complete")
        print(f"   기간         : {start_date} ~ {end_date}")
        print(f"   총 확인      : {total_checked:,}")
        print(f"   신규 INSERT  : {total_inserted:,}")
        print(f"   변경 UPDATE  : {total_updated:,}")
        print(f"   동일 SKIP    : {total_skipped_same:,}")
        print(f"   권한 오류 계정: {total_skipped_perm_accts} {sorted(skipped_accounts)}")
        print(f"{'=' * 60}")

        return {
            "start_date": start_date,
            "end_date": end_date,
            "checked": total_checked,
            "inserted": total_inserted,
            "updated": total_updated,
            "skipped_same": total_skipped_same,
            "skipped_permission_accounts": sorted(skipped_accounts),
        }

    finally:
        conn.close()


# ▼ 실행 (날짜 범위를 직접 지정하거나 위의 START_DATE / END_DATE 환경변수를 사용)
result = run_full_scan("2026-03-29", "2026-03-30")
print(json.dumps(result, ensure_ascii=False, indent=2))


🔍 Full scan: 2026-03-29 ~ 2026-03-30
✅ Acts: 30 | Ads: 3750
📂 Loading DB snapshot (2026-03-29 ~ 2026-03-30)...
   → 0 existing rows loaded

🔄 act_399481461969972: 297 ads
  → inserted=446 updated=0 same=0 checked=446

🔄 act_2031096637697309: 73 ads
  → inserted=556 updated=0 same=0 checked=556

🔄 act_1073418823880031: 158 ads
  → inserted=615 updated=0 same=0 checked=615

🔄 act_4204029286499182: 533 ads
  → inserted=615 updated=0 same=0 checked=615

🔄 act_601392272220010: 72 ads
  ⚠️ Permission error (code=200). 5초 후 재시도...
  ❌ 권한 오류 지속 (act_601392272220010). 계정 전체 스킵.

🔄 act_1202370041938553: 66 ads
  → inserted=823 updated=0 same=0 checked=823

🔄 act_618278251632554: 300 ads
  → inserted=823 updated=0 same=0 checked=823

🔄 act_1174056261489279: 466 ads
  → inserted=1224 updated=0 same=0 checked=1224

🔄 act_1008886398030550: 188 ads
  → inserted=1505 updated=0 same=0 checked=1505

🔄 act_708046804726689: 281 ads
  ⚠️ Permission error (code=200). 5초 후 재시도...
  ❌ 권한 오류 지속 (act_7080468047